# Hunyuan3D-2 Colab Worker: Safe Two-Phase Flow

Run this notebook from top to bottom after stopping old processes.

Safe flow for Colab Free T4:

1. Colab runs the Hunyuan worker at `127.0.0.1:8010`.
2. Ngrok exposes the Colab worker.
3. Windows backend runs FastAPI at `http://<LAN-IP>:8000`.
4. Expo/iPhone calls the Windows backend.
5. Phase 1 default: create shape-only mesh.
6. Phase 2 optional: paint texture later with `/paint-texture` using the saved `job_id`.

Important note: the first texture request may spend several minutes downloading texture weights from Hugging Face if they are not cached yet. Use the optional texture warmup cell before tapping `Paint texture` in Expo.

## 0. Reset Colab processes

In [ ]:
import subprocess, time
for pattern in ['uvicorn colab_hunyuan_worker:app', 'ngrok']:
    subprocess.run(f"pkill -f '{pattern}'", shell=True, check=False)
time.sleep(2)
print('Old worker/ngrok processes killed if they existed.')

## 1. Clone Hunyuan3D-2 official repo

In [ ]:
%cd /content
!rm -rf Hunyuan3D-2
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!git rev-parse --short HEAD

## 2. Install dependencies

Texture modules are still installed, but the default runtime path is shape-only first.

In [ ]:
%cd /content/Hunyuan3D-2
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -e .

# Required only for the later paint-texture phase.
%cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
!python setup.py install
%cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer
!python setup.py install
%cd /content/Hunyuan3D-2

!python -m pip install fastapi uvicorn python-multipart pyngrok requests trimesh psutil

## 3. GPU, RAM, and import check

In [ ]:
import psutil, torch
ram = psutil.virtual_memory()
print('System RAM GB:', round(ram.used / 1024**3, 2), '/', round(ram.total / 1024**3, 2))
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print('GPU VRAM free/total GB:', round(free / 1024**3, 2), '/', round(total / 1024**3, 2))

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
print('shape import: ok')
try:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    print('texture import: ok')
except Exception as exc:
    print('texture import failed:', repr(exc))
    print('Shape-only can still run.')

## 4. Write Colab worker

In [ ]:
from pathlib import Path
worker_path = Path('/content/colab_hunyuan_worker.py')
worker_path.write_text('from __future__ import annotations\n\nimport io\nimport gc\nimport os\nfrom pathlib import Path\n\nfrom fastapi import FastAPI, File, Form, HTTPException, UploadFile\nfrom fastapi.responses import FileResponse\nfrom PIL import Image, ImageOps, UnidentifiedImageError\n\nimport torch\n\n\nWORK_DIR = Path(os.environ.get("HUNYUAN_COLAB_WORK_DIR", "/tmp/hunyuan_jobs"))\nMODEL_ID = os.environ.get("HUNYUAN_MODEL_ID", "tencent/Hunyuan3D-2")\nMODEL_SUBFOLDER = os.environ.get("HUNYUAN_MODEL_SUBFOLDER", "hunyuan3d-dit-v2-0")\nTEXGEN_MODEL_ID = os.environ.get("HUNYUAN_TEXGEN_MODEL_ID", MODEL_ID)\nINFERENCE_STEPS = int(os.environ.get("HUNYUAN_INFERENCE_STEPS", "20"))\nOCTREE_RESOLUTION = int(os.environ.get("HUNYUAN_OCTREE_RESOLUTION", "320"))\nNUM_CHUNKS = int(os.environ.get("HUNYUAN_NUM_CHUNKS", "12000"))\nSEED = int(os.environ.get("HUNYUAN_SEED", "12345"))\nKEEP_SHAPE_PIPELINE = os.environ.get("HUNYUAN_KEEP_SHAPE_PIPELINE", "0").strip().lower() in {\n    "1",\n    "true",\n    "yes",\n    "on",\n}\nKEEP_TEXTURE_PIPELINE = os.environ.get("HUNYUAN_KEEP_TEXTURE_PIPELINE", "0").strip().lower() in {\n    "1",\n    "true",\n    "yes",\n    "on",\n}\n\nWORK_DIR.mkdir(parents=True, exist_ok=True)\napp = FastAPI(title="Hunyuan Colab Worker")\n_shape_pipeline = None\n_texture_pipeline = None\n\n\ndef cleanup_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        try:\n            torch.cuda.ipc_collect()\n        except Exception:\n            pass\n\n\ndef get_shape_pipeline():\n    global _shape_pipeline\n    if _shape_pipeline is None:\n        try:\n            from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline\n        except Exception as exc:\n            raise HTTPException(\n                status_code=503,\n                detail="Hunyuan shape dependencies are not ready. Re-run the install cells and check worker logs.",\n            ) from exc\n\n        cleanup_memory()\n        _shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(\n            MODEL_ID,\n            subfolder=MODEL_SUBFOLDER,\n            use_safetensors=True,\n            device="cuda" if torch.cuda.is_available() else "cpu",\n        )\n    return _shape_pipeline\n\n\ndef get_texture_pipeline():\n    global _texture_pipeline\n    if _texture_pipeline is None:\n        try:\n            from hy3dgen.texgen import Hunyuan3DPaintPipeline\n        except Exception as exc:\n            raise HTTPException(\n                status_code=503,\n                detail=(\n                    "Hunyuan texture dependencies are not ready. Install the official "\n                    "texgen custom_rasterizer and differentiable_renderer modules."\n                ),\n            ) from exc\n\n        cleanup_memory()\n        _texture_pipeline = Hunyuan3DPaintPipeline.from_pretrained(TEXGEN_MODEL_ID)\n    return _texture_pipeline\n\n\ndef unload_shape_pipeline():\n    global _shape_pipeline\n    _shape_pipeline = None\n    cleanup_memory()\n\n\ndef unload_texture_pipeline():\n    global _texture_pipeline\n    _texture_pipeline = None\n    cleanup_memory()\n\n\ndef read_image(image_bytes: bytes) -> Image.Image:\n    try:\n        return ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes))).convert("RGBA")\n    except UnidentifiedImageError as exc:\n        raise HTTPException(status_code=400, detail="Uploaded file is not a valid image.") from exc\n\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "ok",\n        "cuda_available": torch.cuda.is_available(),\n        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",\n        "model_id": MODEL_ID,\n        "model_subfolder": MODEL_SUBFOLDER,\n        "texgen_model_id": TEXGEN_MODEL_ID,\n        "shape_pipeline_loaded": _shape_pipeline is not None,\n        "texture_pipeline_loaded": _texture_pipeline is not None,\n        "cuda_memory_allocated_mb": (\n            round(torch.cuda.memory_allocated() / 1024 / 1024, 1) if torch.cuda.is_available() else None\n        ),\n        "cuda_memory_reserved_mb": (\n            round(torch.cuda.memory_reserved() / 1024 / 1024, 1) if torch.cuda.is_available() else None\n        ),\n    }\n\n\n@app.post("/warmup")\ndef warmup():\n    get_shape_pipeline()\n    return health()\n\n\n@app.post("/warmup-texture")\ndef warmup_texture():\n    get_texture_pipeline()\n    return health()\n\n\nasync def prepare_job_image(image: UploadFile, job_id: str) -> tuple[Image.Image, Path]:\n    image_bytes = await image.read()\n    if not image_bytes:\n        raise HTTPException(status_code=400, detail="Uploaded image is empty.")\n\n    pil_image = read_image(image_bytes)\n    job_dir = WORK_DIR / job_id\n    job_dir.mkdir(parents=True, exist_ok=True)\n    input_path = job_dir / "input.png"\n    pil_image.save(input_path)\n    return pil_image, job_dir\n\n\ndef generate_shape_mesh(pil_image: Image.Image):\n    pipeline = get_shape_pipeline()\n    with torch.inference_mode():\n        return pipeline(\n            image=pil_image,\n            num_inference_steps=INFERENCE_STEPS,\n            octree_resolution=OCTREE_RESOLUTION,\n            num_chunks=NUM_CHUNKS,\n            generator=torch.manual_seed(SEED),\n        )[0]\n\n\ndef validate_glb_output(output_format: str):\n    if output_format.lower() != "glb":\n        raise HTTPException(status_code=400, detail="Only glb output is supported.")\n\n\ndef load_mesh(mesh_path: Path):\n    try:\n        import trimesh\n    except Exception as exc:\n        raise HTTPException(status_code=503, detail=f"trimesh is not importable: {exc}") from exc\n\n    mesh = trimesh.load(mesh_path, force="mesh")\n    if hasattr(mesh, "geometry"):\n        mesh = mesh.dump(concatenate=True)\n    return mesh\n\n\n@app.post("/generate-shape")\nasync def generate_shape(\n    image: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n\n    mesh = generate_shape_mesh(pil_image)\n    output_path = job_dir / "mesh.glb"\n    mesh.export(output_path)\n    del mesh\n    if not KEEP_SHAPE_PIPELINE:\n        unload_shape_pipeline()\n    cleanup_memory()\n    return FileResponse(output_path, media_type="model/gltf-binary", filename="mesh.glb")\n\n\n@app.post("/generate-texture")\nasync def generate_texture(\n    image: UploadFile = File(...),\n    mesh: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n\n    mesh_bytes = await mesh.read()\n    if not mesh_bytes:\n        raise HTTPException(status_code=400, detail="Uploaded mesh is empty.")\n\n    mesh_path = job_dir / "input_mesh.glb"\n    mesh_path.write_bytes(mesh_bytes)\n    input_mesh = load_mesh(mesh_path)\n\n    texture_pipeline = get_texture_pipeline()\n    try:\n        with torch.inference_mode():\n            textured_mesh = texture_pipeline(input_mesh, image=pil_image.convert("RGB"))\n    except RuntimeError as exc:\n        cleanup_memory()\n        raise HTTPException(status_code=500, detail=f"Hunyuan texture generation failed: {exc}") from exc\n\n    output_path = job_dir / "mesh.glb"\n    textured_mesh.export(output_path)\n    del input_mesh\n    del textured_mesh\n    if not KEEP_TEXTURE_PIPELINE:\n        unload_texture_pipeline()\n    cleanup_memory()\n    return FileResponse(output_path, media_type="model/gltf-binary", filename="mesh.glb")\n\n\n@app.post("/generate-textured-shape")\nasync def generate_textured_shape(\n    image: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n\n    mesh = generate_shape_mesh(pil_image)\n    shape_path = job_dir / "shape_mesh.glb"\n    mesh.export(shape_path)\n    if not KEEP_SHAPE_PIPELINE:\n        unload_shape_pipeline()\n    cleanup_memory()\n\n    texture_pipeline = get_texture_pipeline()\n    try:\n        with torch.inference_mode():\n            textured_mesh = texture_pipeline(mesh, image=pil_image.convert("RGB"))\n    except RuntimeError as exc:\n        cleanup_memory()\n        raise HTTPException(status_code=500, detail=f"Hunyuan texture generation failed: {exc}") from exc\n\n    output_path = job_dir / "mesh.glb"\n    textured_mesh.export(output_path)\n    del mesh\n    del textured_mesh\n    if not KEEP_TEXTURE_PIPELINE:\n        unload_texture_pipeline()\n    cleanup_memory()\n    return FileResponse(output_path, media_type="model/gltf-binary", filename="mesh.glb")\n', encoding='utf-8')
print('Wrote:', worker_path)
print('Size:', worker_path.stat().st_size, 'bytes')

## 5. Start worker on port 8010 with low-memory shape settings

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
from google.colab import userdata

os.environ['HUNYUAN_MODEL_ID'] = 'tencent/Hunyuan3D-2'
os.environ['HUNYUAN_MODEL_SUBFOLDER'] = 'hunyuan3d-dit-v2-0'
os.environ['HUNYUAN_TEXGEN_MODEL_ID'] = 'tencent/Hunyuan3D-2'
os.environ['HUNYUAN_KEEP_SHAPE_PIPELINE'] = '0'
os.environ['HUNYUAN_KEEP_TEXTURE_PIPELINE'] = '0'
os.environ['HUNYUAN_INFERENCE_STEPS'] = '15'
os.environ['HUNYUAN_OCTREE_RESOLUTION'] = '256'
os.environ['HUNYUAN_NUM_CHUNKS'] = '6000'

# Optional but strongly recommended for the texture phase.
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN detected from Colab Secrets.')
else:
    print('HF_TOKEN not found. Texture downloads may be slow or rate-limited.')

if 'worker_proc' in globals() and worker_proc.poll() is None:
    worker_proc.terminate()
    time.sleep(2)
subprocess.run("pkill -f 'uvicorn colab_hunyuan_worker:app'", shell=True, check=False)
time.sleep(2)

log_path = Path('/tmp/hunyuan_worker.log')
log_file = log_path.open('w')
worker_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'colab_hunyuan_worker:app', '--app-dir', '/content', '--host', '0.0.0.0', '--port', '8010'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(30):
    time.sleep(1)
    if worker_proc.poll() is not None:
        break
    try:
        import requests
        r = requests.get('http://127.0.0.1:8010/health', timeout=2)
        print('worker pid:', worker_proc.pid)
        print('health:', r.status_code, r.text)
        ready = True
        break
    except Exception:
        pass

print('worker exit code:', worker_proc.poll())
print(log_path.read_text(encoding='utf-8', errors='replace')[-5000:])
if not ready:
    raise RuntimeError('Worker did not answer /health. Read /tmp/hunyuan_worker.log above.')

## 6. Verify local worker health

In [ ]:
import requests
r = requests.get('http://127.0.0.1:8010/health', timeout=20)
print(r.status_code)
print(r.text)
r.raise_for_status()

## 7. Open ngrok tunnel to Colab worker

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import requests, time

ngrok.kill()
time.sleep(2)
ngrok_token = userdata.get('NGROK_TOKEN')
if not ngrok_token:
    raise RuntimeError('Missing Colab secret NGROK_TOKEN')
ngrok.set_auth_token(ngrok_token)

tunnel = ngrok.connect(addr='127.0.0.1:8010', proto='http', bind_tls=True)
PUBLIC_URL = tunnel.public_url.rstrip('/')
print('PUBLIC_URL=', PUBLIC_URL)

r = requests.get(PUBLIC_URL + '/health', headers={'ngrok-skip-browser-warning': 'true'}, timeout=60)
print('remote health:', r.status_code, r.text)
r.raise_for_status()

## 8. Print Windows backend commands: shape-only default

Copy these commands to Windows PowerShell. Restart the backend every time `PUBLIC_URL` changes.

In [ ]:
try:
    url = PUBLIC_URL
except NameError:
    url = 'https://PASTE_PUBLIC_URL_HERE'

print('Run this in Windows PowerShell for PHASE 1: shape-only')
print()
print(r'cd C:\Users\pminh\Desktop\MyProject\AI_3D_Reconstruction_Systerm_TangDien02')
print(r'.\.venv\Scripts\Activate.ps1')
print('$env:RECONSTRUCTION_BACKEND="hunyuan_remote"')
print(f'$env:HUNYUAN_REMOTE_URL="{url}"')
print('$env:HUNYUAN_REMOTE_OUTPUT_FORMAT="glb"')
print('$env:HUNYUAN_REMOTE_ENABLE_TEXTURE="false"')
print('python -m uvicorn server.main:app --host 0.0.0.0 --port 8000')
print()
print('After Expo creates a shape, copy its job_id and run PHASE 2 texture paint:')
print('curl.exe -X POST "http://127.0.0.1:8000/paint-texture" -F "job_id=<JOB_ID_FROM_EXPO_OR_RESPONSE>"')

## 9. Print Expo local commands

In [ ]:
BACKEND_LAN_IP = '192.168.1.6'  # Change this if your Windows LAN IP changes.
BACKEND_PORT = 8000
api_url = f'http://{BACKEND_LAN_IP}:{BACKEND_PORT}'

print('Run this in a second Windows PowerShell window:')
print()
print(r'cd C:\Users\pminh\Desktop\MyProject\AI_3D_Reconstruction_Systerm_TangDien02\mobile')
print(f'$env:EXPO_PUBLIC_API_BASE_URL="{api_url}"')
print('npm install')
print('npm start')
print()
print('iPhone Safari test before Expo:')
print(f'{api_url}/health')

## 10. Direct worker tests through ngrok

In [ ]:
import requests
try:
    url = PUBLIC_URL
except NameError:
    url = input('Paste PUBLIC_URL: ').strip().rstrip('/')
r = requests.get(url + '/health', headers={'ngrok-skip-browser-warning': 'true'}, timeout=60)
print(r.status_code)
print(r.text)
r.raise_for_status()

In [ ]:
# Shape-only direct test through ngrok. This does not involve Windows backend or Expo.
import requests
from pathlib import Path
try:
    url = PUBLIC_URL
except NameError:
    url = input('Paste PUBLIC_URL: ').strip().rstrip('/')

candidates = list(Path('/content/Hunyuan3D-2').rglob('*.png')) + list(Path('/content/Hunyuan3D-2').rglob('*.jpg')) + list(Path('/content/Hunyuan3D-2').rglob('*.jpeg'))
if not candidates:
    raise FileNotFoundError('No test image found. Upload an image and set TEST_IMAGE manually.')
TEST_IMAGE = str(candidates[0])
print('Using image:', TEST_IMAGE)

with open(TEST_IMAGE, 'rb') as f:
    r = requests.post(
        url + '/generate-shape',
        headers={'ngrok-skip-browser-warning': 'true'},
        files={'image': (Path(TEST_IMAGE).name, f, 'image/png')},
        data={'job_id': 'direct_shape_test', 'output_format': 'glb'},
        timeout=1800,
    )
print('status:', r.status_code)
if r.status_code == 200:
    out = Path('/content/direct_shape_test.glb')
    out.write_bytes(r.content)
    print('saved:', out, 'bytes:', out.stat().st_size)
else:
    print(r.text[:4000])
    r.raise_for_status()

## 11. Optional texture warmup before tapping Paint texture in Expo

Run this once after shape-only is proven to work. This pre-downloads and loads the texture pipeline so the first real paint request does not spend minutes fetching Hugging Face files.

If this cell raises a memory error, skip texture on Colab Free for this session.

In [ ]:
import psutil, requests
ram = psutil.virtual_memory()
print('System RAM GB before texture warmup:', round(ram.used / 1024**3, 2), '/', round(ram.total / 1024**3, 2))

r = requests.post('http://127.0.0.1:8010/warmup-texture', timeout=1800)
print(r.status_code)
print(r.text)
r.raise_for_status()

## 12. Optional direct texture test after warmup-texture succeeds

Run this only after the warmup-texture cell is done and local `/health` is still responsive.

In [ ]:
import requests
from pathlib import Path

mesh_path = Path('/content/direct_shape_test.glb')
if not mesh_path.exists():
    raise FileNotFoundError('Run the direct shape-only test first to create /content/direct_shape_test.glb')

img_candidates = list(Path('/content/Hunyuan3D-2').rglob('*.png')) + list(Path('/content/Hunyuan3D-2').rglob('*.jpg')) + list(Path('/content/Hunyuan3D-2').rglob('*.jpeg'))
if not img_candidates:
    raise FileNotFoundError('No test image found under /content/Hunyuan3D-2')
img_path = img_candidates[0]

with open(img_path, 'rb') as image_file, open(mesh_path, 'rb') as mesh_file:
    r = requests.post(
        'http://127.0.0.1:8010/generate-texture',
        files={
            'image': (img_path.name, image_file, 'image/png'),
            'mesh': ('direct_shape_test.glb', mesh_file, 'model/gltf-binary'),
        },
        data={'job_id': 'direct_texture_probe', 'output_format': 'glb'},
        timeout=1800,
    )
print(r.status_code)
if r.status_code == 200:
    out = Path('/content/direct_texture_probe.glb')
    out.write_bytes(r.content)
    print('saved:', out, 'bytes:', out.stat().st_size)
else:
    print(r.text[:4000])
    r.raise_for_status()

## 13. Debug RAM and worker logs

In [ ]:
import psutil, torch
from pathlib import Path
ram = psutil.virtual_memory()
print('System RAM GB:', round(ram.used / 1024**3, 2), '/', round(ram.total / 1024**3, 2))
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print('GPU VRAM free/total GB:', round(free / 1024**3, 2), '/', round(total / 1024**3, 2))
log_path = Path('/tmp/hunyuan_worker.log')
print(log_path.read_text(encoding='utf-8', errors='replace')[-12000:] if log_path.exists() else 'No worker log found')